# 05 PD Model Validation and Improvement

This notebook compares multiple logistic regression Probability of Default model versions and validates whether the Milestone 2 baseline model is reliable enough to support later Expected Loss work.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load Cleaned Dataset

In [ ]:
data_path = "../data/processed/loans_clean_v1.csv"
tables_dir = Path("../reports/tables")
figures_dir = Path("../reports/figures")

tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

loans = pd.read_csv(data_path, low_memory=False)

print(f"Rows: {loans.shape[0]:,}")
print(f"Columns: {loans.shape[1]:,}")

## 1. Purpose of Milestone 3 Validation

This notebook is not trying to build the final best model yet. Its purpose is to validate the baseline logistic regression PD model by checking whether the model remains useful when important feature groups are changed.

The validation focuses on:

- feature overlap
- dependence on LendingClub `grade`, `sub_grade`, and `int_rate`
- calibration stability
- decile separation
- whether the model can still rank credit risk under different feature sets
- whether time-based validation is possible or completed

This matters because the PD model will later feed Expected Loss:

```text
Expected Loss = PD x LGD x EAD
```

For Expected Loss, the model must produce useful probabilities, not only good classification labels. A model with good classification accuracy but poorly calibrated PDs can still create misleading loss estimates.

### Analysis: Milestone 3 Dataset

Milestone 3 continues to use `loans_clean_v1.csv` so that model comparisons are based on the same cleaned population used in Milestone 2. This keeps the validation work focused on feature-set choices and model stability rather than changes in the data sample.

## 3. Define Target and Feature Sets

In [ ]:
target_column = "default_flag"

feature_sets = {
    "A_baseline": {
        "description": "Baseline features with grade, sub_grade, and int_rate",
        "numeric": [
            "loan_amnt", "term", "int_rate", "annual_inc", "dti",
            "delinq_2yrs", "revol_util", "total_acc", "open_acc", "emp_length",
        ],
        "categorical": [
            "grade", "sub_grade", "home_ownership", "verification_status",
            "purpose", "addr_state",
        ],
    },
    "B_remove_sub_grade": {
        "description": "Remove sub_grade, keep grade and int_rate",
        "numeric": [
            "loan_amnt", "term", "int_rate", "annual_inc", "dti",
            "delinq_2yrs", "revol_util", "total_acc", "open_acc", "emp_length",
        ],
        "categorical": [
            "grade", "home_ownership", "verification_status", "purpose", "addr_state",
        ],
    },
    "C_remove_grade_and_sub_grade": {
        "description": "Remove grade and sub_grade, keep int_rate and borrower/loan variables",
        "numeric": [
            "loan_amnt", "term", "int_rate", "annual_inc", "dti",
            "delinq_2yrs", "revol_util", "total_acc", "open_acc", "emp_length",
        ],
        "categorical": [
            "home_ownership", "verification_status", "purpose", "addr_state",
        ],
    },
    "D_borrower_only": {
        "description": "Borrower and loan characteristics only; remove grade, sub_grade, and int_rate",
        "numeric": [
            "loan_amnt", "term", "annual_inc", "dti", "delinq_2yrs",
            "revol_util", "total_acc", "open_acc", "emp_length",
        ],
        "categorical": [
            "home_ownership", "verification_status", "purpose", "addr_state",
        ],
    },
}

required_columns = [target_column]
for feature_set in feature_sets.values():
    required_columns.extend(feature_set["numeric"])
    required_columns.extend(feature_set["categorical"])

required_columns = sorted(set(required_columns))
missing_columns = [column for column in required_columns if column not in loans.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns exist.")

### Why Feature Overlap Matters

`grade`, `sub_grade`, and `int_rate` can overlap conceptually because they all reflect LendingClub's view of borrower risk and loan pricing. If all three are included, the model may rely heavily on LendingClub's existing risk assessment rather than learning from borrower characteristics directly.

The feature sets below test how much model performance changes when these overlapping variables are removed. If performance drops sharply, it suggests that the baseline model may be dominated by LendingClub-assigned risk labels and pricing variables.

## 4. Prepare Modeling Sample and Random Split

In [ ]:
loans_model = loans.dropna(subset=[target_column]).copy()
loans_model[target_column] = loans_model[target_column].astype(int)

y = loans_model[target_column]

train_index, test_index = train_test_split(
    loans_model.index,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Modeling rows: {len(loans_model):,}")
print(f"Training rows: {len(train_index):,}")
print(f"Test rows: {len(test_index):,}")
print(f"Overall default rate: {y.mean():.2%}")
print(f"Training default rate: {loans_model.loc[train_index, target_column].mean():.2%}")
print(f"Test default rate: {loans_model.loc[test_index, target_column].mean():.2%}")

## 5. Helper Functions for Modeling and Evaluation

In [ ]:
def build_logistic_pipeline(numeric_features, categorical_features):
    """Build one logistic regression pipeline with standard preprocessing."""
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    )

    return model


def create_risk_decile_table(y_true, y_pred_proba):
    """Create a risk decile table from actual outcomes and predicted PDs."""
    results = pd.DataFrame({
        "default_flag": y_true.values,
        "predicted_default_probability": y_pred_proba,
    })

    results["risk_decile"] = pd.qcut(
        results["predicted_default_probability"],
        q=10,
        labels=False,
        duplicates="drop",
    ) + 1

    decile_table = (
        results.groupby("risk_decile")
        .agg(
            number_of_loans=("default_flag", "size"),
            average_predicted_pd=("predicted_default_probability", "mean"),
            actual_default_rate=("default_flag", "mean"),
            number_of_defaults=("default_flag", "sum"),
            minimum_predicted_pd=("predicted_default_probability", "min"),
            maximum_predicted_pd=("predicted_default_probability", "max"),
        )
        .reset_index()
        .sort_values("risk_decile")
    )

    return decile_table


def evaluate_feature_set(model_version, feature_set, train_index, test_index, data):
    """Train and evaluate one feature set using the shared train-test split."""
    numeric_features = feature_set["numeric"]
    categorical_features = feature_set["categorical"]
    feature_columns = numeric_features + categorical_features

    X_train = data.loc[train_index, feature_columns]
    X_test = data.loc[test_index, feature_columns]
    y_train = data.loc[train_index, target_column]
    y_test = data.loc[test_index, target_column]

    model = build_logistic_pipeline(numeric_features, categorical_features)
    model.fit(X_train, y_train)

    y_pred_proba = model.predict_proba(X_test)[:, 1]

    decile_table = create_risk_decile_table(y_test, y_pred_proba)
    lowest_default_rate = decile_table.loc[decile_table["risk_decile"].idxmin(), "actual_default_rate"]
    highest_default_rate = decile_table.loc[decile_table["risk_decile"].idxmax(), "actual_default_rate"]

    model_result = {
        "model_version": model_version,
        "feature_set_description": feature_set["description"],
        "number_of_features_after_preprocessing": len(model.named_steps["preprocessor"].get_feature_names_out()),
        "roc_auc": roc_auc_score(y_test, y_pred_proba),
        "pr_auc": average_precision_score(y_test, y_pred_proba),
        "brier_score": brier_score_loss(y_test, y_pred_proba),
        "lowest_decile_default_rate": lowest_default_rate,
        "highest_decile_default_rate": highest_default_rate,
        "highest_lowest_decile_default_rate_ratio": highest_default_rate / lowest_default_rate,
    }

    return model, y_test, y_pred_proba, decile_table, model_result

## 6. Train and Compare Feature Sets

In [ ]:
models = {}
predictions = {}
decile_tables = {}
comparison_results = []

for model_version, feature_set in feature_sets.items():
    model, y_test, y_pred_proba, decile_table, model_result = evaluate_feature_set(
        model_version=model_version,
        feature_set=feature_set,
        train_index=train_index,
        test_index=test_index,
        data=loans_model,
    )

    models[model_version] = model
    predictions[model_version] = {
        "y_test": y_test,
        "y_pred_proba": y_pred_proba,
    }
    decile_tables[model_version] = decile_table
    comparison_results.append(model_result)

comparison_table = pd.DataFrame(comparison_results).sort_values("roc_auc", ascending=False)

comparison_path = tables_dir / "pd_model_feature_set_comparison_v1.csv"
comparison_table.to_csv(comparison_path, index=False)

display(comparison_table)
print(f"Saved feature set comparison table to: {comparison_path}")

## 2. Feature Set Comparison Interpretation

In [ ]:
# Summarize the existing feature set comparison table in a readable format
feature_set_comparison_summary = comparison_table.copy()

summary_metric_columns = [
    "roc_auc",
    "pr_auc",
    "brier_score",
    "lowest_decile_default_rate",
    "highest_decile_default_rate",
    "highest_lowest_decile_default_rate_ratio",
]
feature_set_comparison_summary[summary_metric_columns] = feature_set_comparison_summary[summary_metric_columns].round(4)

feature_set_comparison_summary

In [ ]:
# Identify the best model by each key metric
best_metric_summary = pd.DataFrame({
    "criterion": ["Best ROC-AUC", "Best PR-AUC / Average Precision", "Best Brier Score (lowest)"],
    "model_version": [
        comparison_table.loc[comparison_table["roc_auc"].idxmax(), "model_version"],
        comparison_table.loc[comparison_table["pr_auc"].idxmax(), "model_version"],
        comparison_table.loc[comparison_table["brier_score"].idxmin(), "model_version"],
    ],
    "value": [
        comparison_table["roc_auc"].max(),
        comparison_table["pr_auc"].max(),
        comparison_table["brier_score"].min(),
    ],
})

best_metric_summary["value"] = best_metric_summary["value"].round(4)
best_metric_summary

Feature Set A, the baseline model with `grade`, `sub_grade`, and `int_rate`, has the best overall random-split performance. It has the highest ROC-AUC (**0.7094**), the highest PR-AUC / Average Precision (**0.3701**), and the lowest Brier Score (**0.1453**).

Removing only `sub_grade` in Feature Set B does **not** materially hurt performance. ROC-AUC declines slightly from **0.7094** to **0.7074**, PR-AUC declines from **0.3701** to **0.3686**, and Brier Score worsens only slightly from **0.1453** to **0.1455**. This suggests `sub_grade` adds some signal, but the baseline model does not fully depend on it.

Removing both `grade` and `sub_grade` in Feature Set C also causes only a modest decline. ROC-AUC is **0.7051**, PR-AUC is **0.3647**, and Brier Score is **0.1462**. Because `int_rate` remains in this version, the model still has access to a strong LendingClub pricing signal.

The largest performance drop happens in Feature Set D, where `grade`, `sub_grade`, and `int_rate` are all removed. ROC-AUC falls to **0.6747**, PR-AUC falls to **0.3388**, and Brier Score worsens to **0.1495**. This is a meaningful decline, but the borrower-only model still performs better than random.

**Conclusion:** The baseline model depends meaningfully on `grade`, `sub_grade`, and especially `int_rate`, but borrower-level information still carries useful signal. The model is not purely driven by LendingClub internal risk variables, but those variables clearly improve performance.

## 3. Risk Decile Interpretation

In [ ]:
# Create a compact risk decile interpretation table from the existing comparison results
risk_decile_summary = comparison_table[[
    "model_version",
    "lowest_decile_default_rate",
    "highest_decile_default_rate",
    "highest_lowest_decile_default_rate_ratio",
]].copy()

risk_decile_summary = risk_decile_summary.rename(columns={
    "model_version": "feature_set_name",
    "highest_lowest_decile_default_rate_ratio": "highest_lowest_decile_ratio",
})

risk_decile_summary["decile_pattern_strength"] = pd.cut(
    risk_decile_summary["highest_lowest_decile_ratio"],
    bins=[0, 4, 8, np.inf],
    labels=["weak", "moderate", "strong"],
)

risk_decile_summary[[
    "lowest_decile_default_rate",
    "highest_decile_default_rate",
    "highest_lowest_decile_ratio",
]] = risk_decile_summary[[
    "lowest_decile_default_rate",
    "highest_decile_default_rate",
    "highest_lowest_decile_ratio",
]].round(4)

risk_decile_summary

All four feature sets are useful for ranking borrowers by risk, but they are not equally strong.

Feature Set A has the cleanest and strongest decile separation. Its lowest-risk decile default rate is **4.27%**, while its highest-risk decile default rate is **44.91%**. The highest-risk decile defaults at about **10.52x** the rate of the lowest-risk decile.

Feature Set B remains strong after removing `sub_grade`: lowest decile default rate is **4.63%**, highest decile default rate is **44.77%**, and the ratio is **9.66x**. Feature Set C also remains strong after removing both `grade` and `sub_grade`: lowest decile default rate is **4.66%**, highest decile default rate is **44.34%**, and the ratio is **9.51x**.

Feature Set D is weaker after removing `grade`, `sub_grade`, and `int_rate`. Its lowest decile default rate is **6.92%**, highest decile default rate is **42.03%**, and the ratio falls to **6.08x**. This is still useful separation, but it is clearly weaker than the other feature sets.

The strongest decile ordering comes from Feature Sets A, B, and C. Feature Set D shows weaker separation, but not a complete failure.

**Conclusion:** Yes, the model can consistently separate safer borrowers from riskier borrowers. Separation is strongest when LendingClub grade/pricing variables are included, but borrower-only information still ranks risk meaningfully.

## 7. Calibration Curve Comparison

In [ ]:
plt.figure(figsize=(8, 6))

for model_version, prediction_data in predictions.items():
    prob_true, prob_pred = calibration_curve(
        prediction_data["y_test"],
        prediction_data["y_pred_proba"],
        n_bins=10,
        strategy="quantile",
    )
    plt.plot(prob_pred, prob_true, marker="o", label=model_version)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
plt.xlabel("Average Predicted PD")
plt.ylabel("Actual Default Rate")
plt.title("Calibration Comparison by Feature Set")
plt.legend()
plt.tight_layout()

calibration_comparison_path = figures_dir / "pd_feature_set_calibration_comparison_v1.png"
plt.savefig(calibration_comparison_path, dpi=150)
plt.show()

print(f"Saved calibration comparison chart to: {calibration_comparison_path}")

## 4. Calibration Interpretation

In [ ]:
# Summarize calibration using the existing predictions already created in the notebook
calibration_rows = []

for model_version, prediction_data in predictions.items():
    prob_true, prob_pred = calibration_curve(
        prediction_data["y_test"],
        prediction_data["y_pred_proba"],
        n_bins=10,
        strategy="quantile",
    )

    calibration_table = pd.DataFrame({
        "calibration_bin": range(1, len(prob_true) + 1),
        "average_predicted_pd": prob_pred,
        "actual_default_rate": prob_true,
    })
    calibration_table["prediction_gap"] = (
        calibration_table["average_predicted_pd"]
        - calibration_table["actual_default_rate"]
    )

    calibration_rows.append({
        "model_version": model_version,
        "mean_absolute_calibration_gap": calibration_table["prediction_gap"].abs().mean(),
        "low_risk_bin_gap": calibration_table.loc[
            calibration_table["calibration_bin"] == 1,
            "prediction_gap",
        ].iloc[0],
        "high_risk_bin_gap": calibration_table.loc[
            calibration_table["calibration_bin"] == calibration_table["calibration_bin"].max(),
            "prediction_gap",
        ].iloc[0],
    })

calibration_summary = pd.DataFrame(calibration_rows)
calibration_summary[[
    "mean_absolute_calibration_gap",
    "low_risk_bin_gap",
    "high_risk_bin_gap",
]] = calibration_summary[[
    "mean_absolute_calibration_gap",
    "low_risk_bin_gap",
    "high_risk_bin_gap",
]].round(4)

calibration_summary

Calibration checks whether predicted PDs are close to actual default rates. This is different from ranking performance. A model can rank loans well and still produce probabilities that are too high or too low.

Based on the Brier Score, Feature Set A appears best calibrated overall because it has the lowest Brier Score (**0.1453**). Feature Set B is very close (**0.1455**), Feature Set C is slightly worse (**0.1462**), and Feature Set D is weakest (**0.1495**).

For Feature Set A, the calibration pattern is close in the random split. The low-risk bin is only slightly underpredicted, and the high-risk bin is also only slightly underpredicted. This means the baseline model is not wildly overconfident or underconfident in the random split.

Calibration becomes more concerning when the strongest LendingClub variables are removed. Feature Set D has the weakest Brier Score and therefore the least reliable probability accuracy among the tested versions. The time-validation results also show that Brier Scores worsen on newer loans, which means calibration stability should remain a major focus.

Calibration matters for Expected Loss because PD is directly multiplied by LGD and EAD. If PD is understated in high-risk bands, expected losses will be understated. If PD is overstated in low-risk bands, expected losses may be too conservative.

**Conclusion:** The predicted probabilities are reasonable for a baseline, but calibration improvement should be part of the next milestone before using the PD model as an Expected Loss input.

## 5. Feature Overlap and Business Interpretation

`grade`, `sub_grade`, and `int_rate` may overlap because LendingClub grades and interest rates already reflect borrower risk assessment and loan pricing. These variables are not purely raw borrower characteristics. They partially summarize LendingClub's internal underwriting and pricing view.

The validation results suggest that `sub_grade` adds only modest information beyond `grade` and `int_rate`. Removing `sub_grade` lowers ROC-AUC from **0.7094** to **0.7074**, a small change.

`int_rate` appears to be an important risk proxy. When `grade` and `sub_grade` are removed but `int_rate` remains, Feature Set C still performs close to baseline with ROC-AUC of **0.7051**. When `int_rate` is also removed in Feature Set D, ROC-AUC drops more clearly to **0.6747**.

A model without `grade` and `sub_grade` may be more independent from LendingClub's internal scoring system, but it may also be less accurate. A borrower-only model is more transparent from a raw credit-risk perspective, but it gives up predictive performance.

For an analytical credit risk project, Feature Set A is appropriate as a benchmark because it gives the strongest performance. Feature Sets C and D are also valuable because they show how much independent borrower-level signal exists without relying as heavily on LendingClub's own risk labels. The tradeoff is performance versus independence and interpretability.

## 8. Default Rate by Risk Decile Comparison

In [ ]:
plt.figure(figsize=(10, 6))

for model_version, decile_table in decile_tables.items():
    plt.plot(
        decile_table["risk_decile"],
        decile_table["actual_default_rate"],
        marker="o",
        label=model_version,
    )

plt.xlabel("Predicted Risk Decile")
plt.ylabel("Actual Default Rate")
plt.title("Default Rate by Risk Decile by Feature Set")
plt.xticks(range(1, 11))
plt.legend()
plt.tight_layout()

decile_comparison_path = figures_dir / "pd_feature_set_decile_comparison_v1.png"
plt.savefig(decile_comparison_path, dpi=150)
plt.show()

print(f"Saved decile comparison chart to: {decile_comparison_path}")

## 9. Time-Based Validation

In [ ]:
if "issue_d" in loans_model.columns:
    loans_time = loans_model.copy()
    loans_time["issue_d"] = pd.to_datetime(loans_time["issue_d"], format="%b-%Y", errors="coerce")
    loans_time = loans_time.dropna(subset=["issue_d"]).sort_values("issue_d")

    cutoff_date = loans_time["issue_d"].quantile(0.80)
    time_train_index = loans_time[loans_time["issue_d"] <= cutoff_date].index
    time_test_index = loans_time[loans_time["issue_d"] > cutoff_date].index

    time_validation_results = []

    for model_version, feature_set in feature_sets.items():
        model, y_time_test, y_time_pred_proba, decile_table, model_result = evaluate_feature_set(
            model_version=model_version,
            feature_set=feature_set,
            train_index=time_train_index,
            test_index=time_test_index,
            data=loans_time,
        )

        random_split_row = comparison_table[comparison_table["model_version"] == model_version].iloc[0]

        time_validation_results.append({
            "model_version": model_version,
            "feature_set_description": feature_set["description"],
            "cutoff_date": cutoff_date,
            "time_train_rows": len(time_train_index),
            "time_test_rows": len(time_test_index),
            "time_test_default_rate": y_time_test.mean(),
            "random_split_roc_auc": random_split_row["roc_auc"],
            "time_split_roc_auc": model_result["roc_auc"],
            "random_split_pr_auc": random_split_row["pr_auc"],
            "time_split_pr_auc": model_result["pr_auc"],
            "random_split_brier_score": random_split_row["brier_score"],
            "time_split_brier_score": model_result["brier_score"],
        })

    time_validation_table = pd.DataFrame(time_validation_results)
    time_validation_path = tables_dir / "pd_time_validation_results_v1.csv"
    time_validation_table.to_csv(time_validation_path, index=False)

    display(time_validation_table)
    print(f"Saved time validation results to: {time_validation_path}")
else:
    print("issue_d is not available. Time validation requires keeping issue_d in the processed dataset.")

## 6. Time-Based Validation Interpretation

In [ ]:
# Display the existing time-validation results in a readable format
if (tables_dir / "pd_time_validation_results_v1.csv").exists():
    time_validation_summary = pd.read_csv(tables_dir / "pd_time_validation_results_v1.csv")
    time_validation_display = time_validation_summary.copy()

    time_metric_columns = [
        "time_test_default_rate",
        "random_split_roc_auc",
        "time_split_roc_auc",
        "random_split_pr_auc",
        "time_split_pr_auc",
        "random_split_brier_score",
        "time_split_brier_score",
    ]
    time_validation_display[time_metric_columns] = time_validation_display[time_metric_columns].round(4)
    display(time_validation_display)
else:
    print("Time validation was not possible. issue_d or another origination date must be preserved in the processed dataset.")

Time-based validation was completed because `issue_d` exists in the processed dataset. The split trains on older loans and tests on newer loans, using a cutoff date of **2016-10-01**. The time test set contains **264,013** loans and has a default rate of **21.77%**.

Time-based performance is weaker than random-split performance, but the drop is not catastrophic. For Feature Set A, ROC-AUC falls from **0.7094** to **0.6966**. PR-AUC is stable, moving from **0.3701** to **0.3683**. Brier Score worsens from **0.1453** to **0.1575**, which means probability accuracy is weaker on newer loans.

Feature Sets B and C show similar time-validation behavior. Feature Set D also drops, with ROC-AUC falling from **0.6747** to **0.6632** and Brier Score worsening from **0.1495** to **0.1624**.

This suggests that the models generalize reasonably to newer loans, but calibration and probability accuracy are less stable over time. The higher time-test default rate also indicates that the newer loan population is riskier or different from the random-split population.

**Conclusion:** Time validation supports the model as a baseline, but it is not stable enough to treat as final. Before Expected Loss, the PD model should be checked across issue years and possibly recalibrated for time-period shifts.

## Where the PD Model May Be Wrong

The validation results show several realistic weaknesses.

First, the model may over-rely on LendingClub risk and pricing variables. This is a serious concern because Feature Set D, which removes `grade`, `sub_grade`, and `int_rate`, drops from **0.7094** ROC-AUC to **0.6747** ROC-AUC. The model still works without those fields, but it is clearly weaker.

Second, borrower-only features are less powerful. Feature Set D still has a highest-to-lowest decile ratio of **6.08x**, so borrower-level information is useful, but it does not separate risk as strongly as Feature Set A's **10.52x** ratio.

Third, calibration may be less stable over time. Feature Set A's Brier Score worsens from **0.1453** in the random split to **0.1575** in time validation. This matters directly for Expected Loss.

Fourth, correlated features make interpretation harder. `grade`, `sub_grade`, and `int_rate` likely contain overlapping risk information, so coefficients and performance changes should not be interpreted as clean causal evidence.

Fifth, logistic regression may miss nonlinear patterns or interactions. Borrower risk may not move linearly with variables such as `dti`, `revol_util`, or income.

Finally, the LendingClub dataset contains accepted loans only. It may not represent rejected applicants or a bank's full current credit portfolio. This accepted-loans-only bias limits how broadly the model can be applied.

The most serious current concerns are dependence on LendingClub risk/pricing variables and weaker time-period calibration. Nonlinearity, correlated features, and dataset bias are important future checks.

## Milestone 3 Validation Conclusion

The model is acceptable as a baseline, but not final.

1. **Is the baseline PD model acceptable as a benchmark?**  
Yes. Feature Set A has ROC-AUC of **0.7094**, PR-AUC of **0.3701**, Brier Score of **0.1453**, and strong decile separation with a **10.52x** highest-to-lowest decile default-rate ratio.

2. **Is it good enough to move to model improvement?**  
Yes. The model ranks risk meaningfully and provides a useful benchmark. However, it should move to improvement and validation, not production use.

3. **What are the biggest weaknesses found in validation?**  
The biggest weaknesses are dependence on `grade`, `sub_grade`, and `int_rate`, weaker borrower-only performance, and worse Brier Scores under time-based validation.

4. **What should be improved next?**  
The next step should improve calibration, test additional model types, review feature overlap, and validate stability across time more carefully.

5. **Should the next milestone compare tree-based models?**  
Yes. Tree-based models may capture nonlinear patterns and feature interactions that logistic regression misses. They should be compared carefully against the logistic regression benchmark using the same credit risk metrics.

## Recommended Next Step: Milestone 4

Milestone 4 should focus on improving the PD model. Do not move directly to LGD and EAD until the PD model is better validated.

The next milestone may compare:

- regularized logistic regression
- Random Forest
- Gradient Boosting
- XGBoost or LightGBM if available

The chosen model should be selected based on:

- ROC-AUC
- PR-AUC
- Brier Score
- calibration
- decile separation
- time stability
- interpretability

Do not select the next model only because it has the highest ROC-AUC. For credit risk, the final PD model should rank borrowers well, produce usable probabilities, remain stable over time, and be explainable enough to support Expected Loss analysis.